# PCU-HYBRID-REATTACHMENT-001

Replay the published ranking-only L7/K64 PCU mutation (82.03125% A_eval ranking, 0% greedy direct), then compare the same frozen Granite model with Cell deltas ON, OFF and RESTORED. No CE/readout regularizer is added. Formal seeds are not consumed.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-hybrid-reattachment-001'
REPO = Path('/kaggle/working/mini-cells')
OUT = REPO / 'artifacts/research/pcu-hybrid-reattachment-001/engineering/26090501-l7-k64-ranking-causal-reattach'
REQUIRED_TRANSFORMERS = '5.16.1'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available(), 'GPU required'
registry = json.loads((REPO / 'research/formal_seed_registry.json').read_text())
states = {int(row['seed']): row['state'] for row in registry['seeds']}
assert states == {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu': torch.cuda.get_device_name(0),
    'transformers': transformers.__version__,
    'formal_seed_states': states,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
hf_token = UserSecretsClient().get_secret('HF_TOKEN')
assert hf_token, 'HF_TOKEN is required'
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
login(token=hf_token, add_to_git_credential=False)
print('HF token loaded; value not printed.')


In [ ]:
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001/test_pcu_hybrid_reattachment_001.py'])


In [ ]:
run([sys.executable, 'scripts/research/run_pcu_hybrid_reattachment_001.py', '--device', 'cuda:0'])
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
print(json.dumps(decision, indent=2, sort_keys=True))
print(json.dumps({
    'status': result['status'],
    'ranking_on': result['causal_effect']['ranking_on'],
    'ranking_off': result['causal_effect']['ranking_off'],
    'ranking_gain': result['causal_effect']['ranking_gain'],
    'answer_margin_gain': result['causal_effect']['answer_margin_gain'],
    'B_control_answer_nll_increase': result['causal_effect']['B_control_answer_nll_increase'],
    'base_vs_off_A': result['logit_differences']['base_vs_off_A'],
    'on_vs_restored_A': result['logit_differences']['on_vs_restored_A'],
}, indent=2, sort_keys=True))


A passing engineering status is not a formal scientific PASS. `formal_decision` must remain `RESERVED_UNRUN`, and formal PCU seeds must remain `RESERVED_UNTOUCHED`.
